In [36]:
import java.time.LocalDateTime

fun LongArray.shift(amount: Int = 1) {
    val n = amount.coerceIn(0, size - 1)
    if (n == 0) return

    val retained = sliceArray(0 until size - n)

    for (i in n until size) {
        this[i] = retained[i - n]
    }

    for (i in 0 until n) {
        this[i] = 0
    }
}

class ItemVolumeHistory(val bins: LongArray = LongArray(TOTAL_BINS.toInt())) {

    fun startNextBin() {
        bins.shift(1)
    }

    fun add(amount: Long) {
        bins[0] += amount
    }

    fun getFirstNDays(days: Int): List<List<Long>> {
        if (days < 1) return emptyList()

        val numBinsToday = binsToday()
        val clampedDays = days.coerceIn(1, TOTAL_DAYS - 1)
        val numBins = (clampedDays * BINS_PER_DAY).toInt()
        val today = bins.slice(0 until numBinsToday).reversed()
        val days: MutableList<List<Long>> = bins.slice(numBinsToday until numBins)
            .reversed()
            .chunked(BINS_PER_DAY.toInt()).toMutableList()
        days += today
        return days
    }

    private fun binsToday(): Int {
        val midnight = LocalDateTime.now()
            .withHour(0)
            .withMinute(0)
            .withSecond(0)

        val now = LocalDateTime.now()
        val minsSinceMidnight = ChronoUnit.MINUTES.between(midnight, now)
        return ceil(minsSinceMidnight / RESOLUTION.toDouble()).toInt()
    }
}

val RESOLUTION = 15L
val BINS_PER_DAY = 24 * (60 / RESOLUTION)
val TOTAL_DAYS = 90
val TOTAL_BINS = TOTAL_DAYS * BINS_PER_DAY

In [17]:
import me.danny.shop.tracking.Graph
import java.util.concurrent.TimeUnit
import net.md_5.bungee.api.ChatColor

fun printGraph(points: List<Long>) {
    val graph = Graph.create(points = points, resolution = 1, timescale = TimeUnit.DAYS)
    graph.forEach {
        println(ChatColor.stripColor(it))
    }
}

In [39]:
fun fakeData(floor: Long, ceiling: Long): ItemVolumeHistory {
    val bins = ItemVolumeHistory()
    (0 until TOTAL_BINS).forEach {
        bins.startNextBin()
        bins.add((floor..ceiling).random())
    }

    return bins
}

In [47]:
val hist = fakeData(0, 100)
val week = hist.getFirstNDays(90).map(List<Long>::sum)
printGraph(week)

▇╱╱╱╱╱▃╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱▃╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱╱▃╱╱╱╱╱╱╱╱╱╱╱ 5.38k
▇╱╱╱▇╱▇╱╱╱▃▇╱╱▃╱╱╱▃╱╱▇▇▇╱╱▇╱▃╱╱▇▇▇╱▇▇╱╱▃▇╱▃▇╱╱▃▃▃▃╱▃▇╱▃╱▃╱╱╱╱╱╱▇▃╱╱╱╱▃╱╱▃▃▃▇▇╱▇▇╱╱▃▃╱▇▃▃▃╱╱ 
▇╱╱▃▇▇▇▃╱▇▇▇╱▇▇▃╱▇▇▇╱▇▇▇▇╱▇▃▇▇▃▇▇▇▇▇▇▇▇▇▇▃▇▇▇▇▇▇▇▇▇▇▇╱▇▇▇▃╱▇▃▇╱▇▇▇▃▇▃▇▇╱▇▇▇▇▇▃▇▇▃▇▇▇▃▇▇▇▇╱╱ 
▇▃▇▇▇▇▇▇▃▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇╱╱ 
▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇╱╱ 3.84k
▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇╱╱ 
▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇╱▇ 
▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇╱▇ 
▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇╱▇ 
▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇╱▇ 1.92k
▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇